In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import torch
assert torch.cuda.is_available(), 'no GPU: switch the runtime to a GPU and rerun'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/temporal-ood-ddi'
os.makedirs(f'{D}/runs/preds', exist_ok=True)
os.makedirs(f'{D}/weights', exist_ok=True)
print(D)

In [ ]:
!mkdir -p /content/w && cp {D}/rev2_bundle.tgz /content/rev2_bundle.tgz && tar xzf /content/rev2_bundle.tgz -C /content/w && cp {D}/inputs.json.gz /content/w/
!pip -q install accelerate sentencepiece protobuf
%cd /content/w

In [ ]:
JOBS = [
 "r2-b2sentence-biomedbert-s42",
 "r2-b2sentence-biomedbert-s1",
 "r2-b2sentence-biomedbert-s2",
 "r2-b2sentence-biomedbert-s3",
 "r2-b2sentence-biomedbert-s4",
 "r2-marker-biomedbert-s42",
 "r2-marker-biomedbert-s1",
 "r2-marker-biomedbert-s2",
 "r2-marker-biomedbert-s3",
 "r2-marker-biomedbert-s4"
]

In [ ]:
%cd /content/w
!bash -c "set -o pipefail; python -m ddiood.rev2_jobs run --workdir /content/w --out {D}/runs --weights {D}/weights --only {' '.join(JOBS)} 2>&1 | tee -a {D}/runs/r2-R2a_sentence_marker.log | (grep --line-buffered -E '^(start|done|skip|FAILED)|Error|Traceback'; true)"

In [ ]:
import os
done = sorted(j for j in JOBS if os.path.exists(f'{D}/runs/preds/{j}.json'))
failed = sorted(j for j in JOBS if os.path.exists(f'{D}/runs/preds/{j}.failed.txt'))
print(f'{len(done)}/{len(JOBS)} jobs finished')
print('missing:', [j for j in JOBS if j not in done])
print('failed:', failed)
for j in failed:
    print(j, open(f'{D}/runs/preds/{j}.failed.txt').read()[-1500:])
assert not failed, failed
assert len(done) == len(JOBS), [j for j in JOBS if j not in done]
no_weights = [j for j in done if not os.path.isdir(f'{D}/weights/{j}')]
print('finished without saved weights:', no_weights)
assert not no_weights, no_weights

In [ ]:
from google.colab import drive
drive.flush_and_unmount()